# 04 — Count cells inside ROIs you draw by hand

The manual counterpart to `01_calibrate` / `02_batch`. There is **no atlas here**: no ABBA,
no registration, no whole-brain map. You open an image in QuPath, draw ROIs, and count
inside them — using the same detection, measurement and classification machinery the
registered route uses.

**The loop is two apps, and QuPath is the one that matters.**

| where | what |
|---|---|
| **QuPath** | draw ROIs → run `roi_count.groovy` → **look at the overlay** |
| **here** | set the two cuts by eye, read the counts, check what is poolable |

### What the operator sees is the ground truth

Every number this notebook prints traces back to detections you can put on the image with
the overlay on. If the DAPI segmentation looks wrong, or a cell called `TdT+` plainly is
not one, then it *is* wrong — whatever the numbers, bands or seed values say. Change the
settings and re-count. Do not reinterpret the picture to match the table.

That is why the two cuts that decide everything each get a section here where you set them
**by looking**, not by reasoning about intensities:

- §3 the **anchor cut** — which pixels are nucleus
- §4 the **marker cut** (`k`) — which cells are positive

### Acquisition varies, so settings are per image

Images counted this way may differ drastically in magnification, Z handling and intensity.
`roi_count.groovy` therefore stores settings **per image** in `roi_settings.yml` and stamps
a `settings_hash` on every count row. §5 tells you which images share a rule. Nothing is
ever blocked — the call is yours — but two densities in the same units look comparable
whatever produced them, so the check is not optional reading.

## PARAMS — the only cell you edit

In [ ]:
PARAMS = {
    # QuPath project holding pipeline.yml, scripts/ and results/roi/.
    # Relative to the repo root (discovered below); an absolute path also works.
    "project": "ROI counting/ROI QuPath",

    # Image to set cuts against in §3/§4. None -> the first one found in the project.
    # Any .ome.tiff path works; this route does not require a MIP naming convention.
    "image": None,

    # "pooled": one row per ROI NAME (three shapes named LA -> one LA row).
    # "shape" : one row per drawn shape, so you can see between-shape spread.
    "scope": "pooled",

    # Marker roles for the engram metrics. None -> resolved from pipeline.yml
    # compartments (whole-cell/cytoplasmic = tagged, nuclear = activity).
    "tagged_marker": None,
    "activity_marker": None,
}

import sys
from pathlib import Path

# Repo root is DISCOVERED, not hardcoded -- a new user's first act should not be editing
# an absolute path with someone else's username in it.
def _find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "scripts" / "cockpit_roi.py").is_file() and (d / "pipeline.yml").is_file():
            return d
    raise FileNotFoundError(
        f"could not find the pipeline repo root at or above {start} -- expected a "
        f"directory holding both scripts/cockpit_roi.py and pipeline.yml. Launch "
        f"JupyterLab from inside the repo, or set REPO = Path('...') by hand here."
    )


REPO = _find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO / "scripts"))


def _under_repo(p) -> Path:
    p = Path(p)
    return p if p.is_absolute() else REPO / p

# ipykernel in this env does not auto-activate the inline backend, so matplotlib falls
# back to the non-interactive Agg canvas and plt.show() silently renders nothing.
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

import cockpit_roi as croi

PROJECT = _under_repo(PARAMS["project"])
ROI_DIR = croi.roi_dir(PROJECT)
print(f"project  : {PROJECT}")
print(f"roi out  : {ROI_DIR}   ({'exists' if ROI_DIR.is_dir() else 'not created yet'})")

## 1. Is this project set up? &nbsp;<sub>CHEAP</sub>

A manual-ROI project needs far less than a registered one: **no `BraiAn.yml`, no atlas, no
registration**. Just `pipeline.yml` (which markers exist and which compartment each is
measured on) and `scripts/roi_count.groovy`.

`roi_settings.yml` writes itself on the first run — you do not create it.

In [ ]:
config = None
try:
    config = croi.load_config(PROJECT)
    print(f"markers  : anchor={config.anchor_name}, "
          + ", ".join(f"{m['name']}/{m['channel']}/{m['compartment']}" for m in config.markers))
    print(f"Double+  : {'yes' if config.emit_double else 'no (needs >=2 markers)'}")
except FileNotFoundError as exc:
    print(f"MISSING  : {exc}")
    print("           Copy the repo-root pipeline.yml into the project and edit the channel")
    print("           names to match this image's channels, or run:")
    print(f'             python3 scripts/sync_project.py --project "{PROJECT}"')

script = PROJECT / "scripts" / "roi_count.groovy"
print(f"script   : {'deployed' if script.is_file() else 'NOT deployed'}  ({script})")
if not script.is_file():
    print(f'           python3 scripts/sync_project.py --project "{PROJECT}"')

settings = PROJECT / "roi_settings.yml"
print(f"settings : {'present' if settings.is_file() else 'not yet — written on the first run'}")

## 2. Count — this happens in QuPath &nbsp;<sub>EXPENSIVE, and not scriptable from here</sub>

This notebook deliberately does **not** drive QuPath. Counting is where you look at the
image, and automating the looking away is the one thing that would break the method.

1. Open the image in QuPath (`File > Open`, or the project entry).
2. Draw ROIs with any area tool — rectangle, polygon, brush, wand.
3. **Name them** in the Annotations pane if you want named regions (`LA`, `CA1`, …).
   Same name on several shapes ⇒ they also pool into one row. Unnamed shapes become
   `ROI_1`, `ROI_2`, … and the name is written back so QuPath and the CSV agree.
4. `Automate > Script editor` → open `scripts/roi_count.groovy` → **Run**.
5. A settings dialog appears, seeded from this image's saved block. Adjust, OK.
6. **Look at the result.** Toggle the detection overlay. Zoom in.

Selecting annotations first restricts counting to the selection; selecting nothing counts
every top-level area annotation on the image.

**Re-running is cheap and safe.** Detections inside the target ROIs are cleared and
rebuilt; nothing outside them is touched. The dialog's **Stage** control matters:

| stage | cost | when |
|---|---|---|
| `Detect + classify + export` | minutes | segmentation changed |
| `Classify + export only` | seconds | only `k` / a marker cut changed |
| `Export only` | instant | just rewriting the CSVs |

So tuning `k` never costs a detection run.

## 3. The anchor cut, set by LOOKING &nbsp;<sub>CHEAP — interactive</sub>

Which pixels count as nucleus. Scrub the slider and watch the mask land on nuclei or bleed
into background; stop when it looks right.

`roi_count.groovy` offers three modes and **prints all three candidate values every run**,
so you can see them disagree before committing:

| mode | what it does |
|---|---|
| `image_span` | `floor + span_frac x (bright - floor)` from the **whole image** histogram — the registered route's rule, so ROI counts stay comparable with atlas counts on the same image |
| `roi_span` | the same rule, measured **only inside your ROIs**. Often the right one when the ROIs sit on tissue and most of the frame does not |
| `absolute` | a number you set here, by eye |

By this project's evidence hierarchy a cut you set while looking at the mask is **tier-1
(SEEN)**, and the span rule is tier-3 — seeded from one operator call on one section and
never validated against hand counts. On the registered route it misfired or looked suspect
on 5 of 16 sections (31%). At an unfamiliar magnification expect that to be worse, not
better. **Setting it by eye per image is the primary method here, not a fallback.**

Paste the value you settle on into the dialog's *Absolute cut* field with mode `absolute`
— or leave a span mode selected if the automatic cut already matches what you see.

In [ ]:
# CHEAP — reads the image directly. No QuPath, no JVM, nothing is written.
import cockpit_threshold_gui as tgui

IMAGE = PARAMS["image"]
if IMAGE is None:
    found = tgui.find_mips(PROJECT)
    IMAGE = found[0] if found else None
    print(f"image: {IMAGE}" if IMAGE else
          "image: none found beside the project — set PARAMS['image'] to a file path")

THRESHOLD = tgui.launch(mip=IMAGE, anchor=(config.anchor_name if config else "DAPI")) if IMAGE else None

# THRESHOLD is a live dict — read the value back rather than retyping it:
#     THRESHOLD["threshold"], THRESHOLD["method"], THRESHOLD["span_frac"]

## 4. The marker cut (`k`), set by LOOKING &nbsp;<sub>CHEAP — interactive</sub>

Which cells are positive. Cell centroids are ringed on the marker channel — filled where
the cell is positive at the current `k`, hollow where it is not. Move `k` until the filled
rings are the cells you would have called positive yourself.

This reads the per-cell export already on disk, so it needs **no re-detection**: changing
`k` is seconds. Whatever you settle on goes into the dialog's `k for <marker>` field
(or `…or absolute <marker> bg-sub cut` if you would rather fix the cut outright).

Two things worth holding while you look:

- **TdT is measured on the whole cell**, so a passing axon can light up a bystander
  nucleus. That inflates `TdT+` and biases reactivation *conservative*. Raising `k` or
  lowering `cell_expansion_um` are the levers.
- The cut is always derived from **the whole image's** classifiable cells, not the crop
  you happen to be looking at — otherwise the number would change as you scroll.

In [ ]:
# CHEAP — reads *__percell_export.tsv from results/roi/. No QuPath, no re-detection.
import cockpit_marker_gui as mgui

K = None
if not ROI_DIR.is_dir() or not list(ROI_DIR.glob("*__percell_export.tsv")):
    print(f"no per-cell exports in {ROI_DIR} yet — run roi_count.groovy on an image first (§2).")
else:
    # results_dir points the picker at the ROI exports; the schema is identical to the
    # registered route's, so the picker itself needed no changes.
    K = mgui.launch(PROJECT, results_dir=ROI_DIR)

# Crop previews need the image findable in a mips/ dir beside the project. Without it the
# picker still shows the distribution and the cut — and QuPath's own overlay, which you
# already have open, is the better picture anyway.

## 5. What is poolable &nbsp;<sub>CHEAP</sub>

The check that stops a magnification difference from being read as biology.

Every count row carries a `settings_hash` — a digest of the *rule* that produced it. It
deliberately **excludes** the resolved threshold under the self-calibrating modes (each
image getting its own number is exactly what makes them comparable) and **includes**
pixel size (a different pixel size is a different measurement, not a rescaling).

One group ⇒ pool freely. More than one ⇒ the report names which fields differ, with
geometry breaks (`pixel_um`, `n_z_planes`, channel) listed first because those are the
ones no threshold rule can absorb.

Below that, the acquisition table re-expresses the micron settings **in each image's own
pixels** — the thing you cannot see in a spreadsheet, and the thing that decides whether
the segmentation behind these counts could have worked at all.

In [ ]:
counts = croi.load_combined(PROJECT)
if counts.empty:
    print(f"no counts under {ROI_DIR} — run roi_count.groovy first (§2).")
else:
    croi.print_comparability(counts)

In [ ]:
if not counts.empty:
    acq = croi.acquisition_table(counts)
    show = [c for c in ("image", "pixel_um", "nucleus_10um_px", "sigma_px", "min_area_px",
                        "expansion_px", "anchor_threshold", "threshold_mode", "advisories")
            if c in acq.columns]
    with pd.option_context("display.max_columns", None, "display.width", 200,
                           "display.max_colwidth", 70):
        display(acq[show])

## 6. The counts &nbsp;<sub>CHEAP</sub>

Per ROI: raw counts, densities, and the locked engram metric family — imported from
`cockpit_animal`, not recomputed here, so these are the same definitions the registered
route reports.

**Read `overlap_above_chance`, not the raw `Double+/TdT+` ratio.** The raw ratio inflates
wherever the activity marker is dense: a region can be 69% `Fos+` among `TdT+` cells and
still be only 1.4x above chance, while a sparser region at a lower raw rate is 4x. Chance
correction is what makes two ROIs comparable.

In [ ]:
readout = pd.DataFrame()
if not counts.empty:
    readout = croi.wide(counts, scope=PARAMS["scope"])
    if config is not None:
        try:
            readout = croi.add_metrics(readout, config,
                                       tagged=PARAMS["tagged_marker"],
                                       activity=PARAMS["activity_marker"])
        except ValueError as exc:
            print(f"metrics skipped: {exc}")

    drop = set(croi.PROVENANCE_FIELDS) | {"anchor_threshold"}
    show = [c for c in readout.columns if c not in drop]
    with pd.option_context("display.max_columns", None, "display.width", 220):
        display(readout[show])

In [ ]:
# Counts and above-chance overlap, side by side. Bars are coloured by settings_hash, so a
# group that was counted by a different rule is visibly a different colour rather than
# quietly sitting in the same series.
if not readout.empty and "overlap_above_chance" in readout.columns:
    import numpy as np

    d = readout.dropna(subset=["overlap_above_chance"]).copy()
    d["label"] = d["image"].astype(str) + " / " + d["roi_name"].astype(str)
    d = d.sort_values("overlap_above_chance", ascending=True)

    hashes = list(dict.fromkeys(d["settings_hash"]))
    palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    colors = [palette[hashes.index(h) % len(palette)] for h in d["settings_hash"]]

    fig, axes = plt.subplots(1, 2, figsize=(13, max(2.5, 0.42 * len(d))), sharey=True)
    anchor_col = f"{config.anchor_name}_density"
    if anchor_col in d.columns:
        axes[0].barh(d["label"], d[anchor_col], color=colors)
        axes[0].set_xlabel(f"{config.anchor_name} per mm²")
        axes[0].set_title("density")

    # log2 on a LINEAR axis: chance sits at 0, so a bar has an honest baseline. A raw
    # ratio has its baseline at 1 and no honest zero, which is why it is not plotted.
    axes[1].barh(d["label"], np.log2(d["overlap_above_chance"]), color=colors)
    axes[1].axvline(0, color="0.3", lw=1)
    axes[1].set_xlabel("log2(overlap_above_chance)   —   0 = chance")
    axes[1].set_title("overlap above chance")

    if len(hashes) > 1:
        handles = [plt.Rectangle((0, 0), 1, 1, color=palette[i % len(palette)])
                   for i in range(len(hashes))]
        axes[1].legend(handles, hashes, title="settings_hash", fontsize=8, loc="lower right")
        fig.suptitle("colours are different counting rules — compare across them deliberately",
                     fontsize=9, y=1.02)
    fig.tight_layout()
    plt.show()
elif not readout.empty:
    print("no overlap_above_chance column — that needs two markers and a Double+ count.")

## 7. Save the readout &nbsp;<sub>CHEAP</sub>

One tidy CSV, provenance included. Keep the provenance columns: they are what lets someone
(including you, later) tell whether two rows were made the same way.

In [ ]:
if not readout.empty:
    out = ROI_DIR / f"roi_readout_{PARAMS['scope']}.csv"
    readout.to_csv(out, index=False)
    print(f"wrote {out}  ({len(readout)} rows)")
    print()
    print("Before this leaves the notebook: are the ROIs still what you meant, and did the")
    print("overlay look right in QuPath? Those two questions outrank everything above.")